In [1]:
# защита от падения ядра IPython
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

In [2]:
import numpy as np
from pathlib import Path
import pandas as pd
import shutil

from src.data_preprocessing import (
    read_oscilloscope_data,
    segment_signal,
    perform_fft_on_segments,
    spectrogram_samples_manual
)

Набираем датасет.

Для engine_1:

In [ ]:
def create_engine_1_data(input_path: str,
                          count_1d: int = 2000, 
                          count_2d: int = 8000, 
                          output_path: str = '../data/processed',
                          input_data_type: str = 'txt',
                          segment_length=10000,
                          f_sampling=10000,
                          step=20,
                          apply_window='blackman'):
    '''
    Function for collecting 1D-spectra and spectrograms

    Parameters:
    - input_path: raw data folder
    - count_1d: number of 1D-spectra for each signal type
    - count_2d: number of spectrograms for each signal type
    - output_path: processed data folder (inside output_path, a folder named 'ours_FFT' 
    - input_data_type: using .txt or .csv data
    for 1D-spectra and a folder named 'ours_spectrogram' for spectrograms are created)
    '''
    if os.path.isdir(output_path):
        shutil.rmtree(output_path)
    os.makedirs(output_path)
    for input_file in Path(input_path).rglob(f'*.{input_data_type}'):
        if input_data_type == 'txt':
            open(f'{output_path}/Anomalous.csv', 'a').close()
            df = read_oscilloscope_data(
                file_path=input_file,
                output_path=f'{output_path}/Anomalous.csv'
            )
            some_signal = df['Data'].to_numpy()
        else:
            df = pd.read_csv(input_file)
            some_signal = df['1'].fillna(0).to_numpy()
        some_signal_mean = np.mean(some_signal)
        some_signal -= some_signal_mean
        segments = segment_signal(some_signal, segment_length=segment_length, step=step, apply_window=apply_window)[:count_1d]
        fft_segments, freqs = perform_fft_on_segments(segments, f_sampling=f_sampling, db=True, cutoff_freq=250)
        os.makedirs(f"{output_path}/{input_file.name}/ours_FFT")
        os.makedirs(f"{output_path}/{input_file.name}/ours_spectrogram")
        for idx in range(len(fft_segments)):
            np.save(f"{output_path}/{input_file.name}/ours_FFT/{idx}.npy", fft_segments[idx])
        signal_samples = spectrogram_samples_manual(some_signal, 3, count_2d, f_sampling=f_sampling)
        for idx in range(len(signal_samples)):
            np.save(f"{output_path}/{input_file.name}/ours_spectrogram/{idx}_FFT.npy", signal_samples[idx][3])


create_engine_1_data(input_path='../dataset/engine_1', 
                     output_path='../data/processed_engine_1')

Для engine_2:

In [ ]:
def create_engine_2_data(input_path: str,
                          count_1d: int = 2000, 
                          count_2d: int = 2000, 
                          output_path: str = '../data/processed',
                          input_data_type: str = 'csv',
                          segment_length=4098,
                          f_sampling=4098,
                          step=20,
                          apply_window='blackman'):
    if os.path.isdir(output_path):
        shutil.rmtree(output_path)
    os.makedirs(output_path)
    engine_2_signals = Path(input_path).glob('*')
    for load_folder in engine_2_signals:
        phases = load_folder.glob("*")
        for phase_folder in phases:
            signal_file = list(phase_folder.glob(f'*.{input_data_type}'))[0]
            if input_data_type == 'csv':
                df = pd.read_csv(signal_file)
                engine_2_signal = df['1'].fillna(0).to_numpy()
            else:
                open(f'{output_path}/Anomalous.csv', 'a').close()
                df = read_oscilloscope_data(
                    file_path=signal_file,
                    output_path=f'{output_path}/Anomalous.csv'
                )
                engine_2_signal = df['Data'].to_numpy()
            engine_2_signal_mean = np.mean(engine_2_signal)
            engine_2_signal -= engine_2_signal_mean
            engine_2_segments = segment_signal(engine_2_signal, 
                                               segment_length=segment_length, 
                                               step=step, 
                                               apply_window=apply_window)[:count_1d]
            engine_2_fft, engine_2_freqs = perform_fft_on_segments(engine_2_segments, 
                                                                   f_sampling=f_sampling, 
                                                                   db=True, 
                                                                   cutoff_freq=250)
            full_output_path = f'{output_path}/{load_folder.stem}/{phase_folder.stem}/{signal_file.stem}'
            os.makedirs(f"{full_output_path}/ours_FFT")
            os.makedirs(f"{full_output_path}/ours_spectrogram")
            for idx in range(len(engine_2_segments)):
                np.save(f"{full_output_path}/ours_FFT/{idx}.npy", engine_2_fft[idx])
            signal_samples = spectrogram_samples_manual(engine_2_signal, 3, count_2d, f_sampling=f_sampling)
            for idx in range(len(signal_samples)):
                np.save(f"{full_output_path}/ours_spectrogram/{idx}_FFT.npy", signal_samples[idx][3])


create_engine_2_data(input_path='../dataset/engine_2/experiment_1/current', 
                     output_path='../data/processed_engine_2')                